### Imports & EC3 Python Wrapper Setup

In [1]:
import os
import pandas as pd
import json
from pprint import pprint
from ec3 import EC3epds #Requires ec3-python-wrapper package

token = os.environ['EC3_KEY'] #assumes EC3 access token is stored as environment variable
ec3_epds = EC3epds(bearer_token=token, ssl_verify=False)

### Main EC3 Query
**This will take some time to run if max_records is not limited**

In [16]:
#Getting material records example
from ec3 import EC3Materials
ec3_materials = EC3Materials(bearer_token=token, ssl_verify=False)

ec3_materials.return_fields = ["id",
                              "open_xpd_uuid",
                              "created_on",
                              "updated_on",
                              "postalCode",
                              "date_of_issue",
                              "cementitious",
                              "standard_deviation",
                              "uncertainty_factor",
                              "concrete_compressive_strength_28d",
                              "concrete_aggregate_size_max",
                              "gwp",
                              "gwp_per_category_declared_unit",
                              "plant_geography",
                              "lightweight",
                              "plant_or_group",
                              "latitude",
                              "longitude",
                              "name"]


#ec3_materials.sort_by = "concrete_compressive_strength_28d" #This will sort the responses based on the field assiged to the 'sort_by' property
ec3_materials.only_valid = True
#ec3_materials.max_records = 50

#List of filters to apply to the material search
mat_filters = [
    # {
    #   "field": "concrete_compressive_strength_at_28d",
    #   "op": "gt",
    #   "arg": "3500 psi"
    # },
    # {
    #   "field": "concrete_compressive_strength_at_28d",
    #   "op": "lte",
    #   "arg": "4500 psi"
    # },
    {
      "field": "lightweight",
      "op": "exact",
      "arg": False
    },
    {
      "field": "plant_geography",
      "op": "in",
      "arg": ["US"]
    },
    {
      "field": "manufacturer_specific",
      "op": "exact",
      "arg": True
    },
    {
      "field": "plant_specific",
      "op": "exact",
      "arg": True
    },
    # {
    # "field": "cementitious__fly_ash",
    # "op": "gt",
    # "arg": 0
    # }
  ]

scm_types = ["ggbs", "gg45", "fly_ash", "nat_poz", "opc", "mk", "other"]
all_records = []

for scm in scm_types:
    mat_filters_scm = mat_filters.copy()
    mat_filters_scm.append({
        "field": f"cementitious__{scm}",
        "op": "gt",
        "arg": 0
    })
    records = ec3_materials.get_materials_mf("ReadyMix", mat_filters_scm, return_all=True)
    all_records.extend(records)


# Remove duplicates based on ID
unique_records = list({record['id']: record for record in all_records}.values())


# #NOTE The following query may take a few minutes to return all responses depending on parameters passed
# #Setting return_all to True will ignore the max_records number and attempt to return all matches
# mat_records = ec3_materials.get_materials_mf("ReadyMix", mat_filters, return_all=False)

print(len(unique_records))
# pprint(mat_records[0])


5037


In [13]:
pprint(unique_records[10])

{'cementitious': {'fly_ash': 0.15, 'ggbs': 0.35},
 'concrete_compressive_strength_28d': '24.1 MPa',
 'created_on': '2024-04-26T19:53:12.535735Z',
 'gwp': '271 kgCO2e',
 'gwp_per_category_declared_unit': '271 kgCO2e',
 'id': '5f3ff9387a6a4276aa532603f058533e',
 'lightweight': False,
 'name': 'Mix 67T550',
 'open_xpd_uuid': 'ec376bcm',
 'plant_or_group': {'created_on': '2022-05-10T20:37:46.919467Z',
                    'id': 'b2bf3a7f537c4c3791894e55fd87eb59',
                    'latitude': 36.9834654,
                    'longitude': -122.0323577,
                    'name': 'Santa Cruz',
                    'type': 'Plant',
                    'updated_on': '2024-10-15T20:00:30.102414Z'},
 'standard_deviation': '30.45345544 kgCO2e',
 'uncertainty_factor': 1.0945742734134394,
 'updated_on': '2024-05-24T16:20:51.514550Z'}


In [ ]:
def filter_records(epd_list):
    '''
    The returned records contain lots of fields we don't need.
    This strips them down prior to saving to json to make a more workable file size.
    '''
    filt_epd_list = []
    for epd in epd_list:
        new_dict = {}
        new_dict['id'] = epd['id']
        new_dict['epd_id'] = epd['epd_id']
        new_dict['open_xpd_uuid'] = epd['open_xpd_uuid']
        new_dict['description'] = epd['description']
        new_dict['date_of_issue'] = epd['date_of_issue']
        new_dict['compressive_strength'] = epd.get('concrete_compressive_strength_28d')
        new_dict['gwp'] = epd.get('gwp')
        new_dict['gwp_per_category_declared_unit'] = epd.get('gwp_per_category_declared_unit')
        new_dict['plant_geography'] = epd.get('plant_or_group')
        new_dict['plant_country'] =  new_dict['plant_geography'][0][0:2]
        new_dict['plant_subdiv'] = new_dict['plant_geography'][0][-2:]

        filt_epd_list.append(new_dict)

    return filt_epd_list

In [3]:
ec3_epds = EC3epds(bearer_token=token, ssl_verify=False)

epd_param_dict = {"concrete_compressive_strength_28d__gt":"3500 psi",
                  "concrete_compressive_strength_28d__lte":"4500 psi",
                  "lightweight":False,
                  "plant_geography": ["US"],
                  "product_specific": True,
                  "manufacturer_specific": True,
                  "item.cement_scm__in": ["ggbs", "flyAsh", "natPoz", "siFume", "gg45", "mk", "CaC03", "other"],
                  #"cementitious__fly_ash__gt": 0,
                  }

ec3_epds.display_name_filter = ["Ready Mix"]

ec3_epds.only_valid = False

ec3_epds.return_fields = ["id",
                          "open_xpd_uuid",
                          "date_of_issue",
                          "cementitious",
                          "concrete_compressive_strength_28d",
                          "gwp", "gwp_per_category_declared_unit",
                          "plant_geography",
                          "lightweight",
                          #"plant_or_group.name"
                          ]
#ec3_epds.sort_by = "date_of_issue"
ec3_epds.page_size = 100

epd_records = ec3_epds.get_epds(return_all=False, params=epd_param_dict)

print(epd_records[0])

{'gwp': '327 kgCO2e', 'plant_geography': ['US-CO'], 'short_link': 'buildingtransparency.org/e/ec399r0r8r', 'lightweight': False, 'date_of_issue': '2024-07-25', 'id': '4f33d368eba2431d97824e89933fa015', 'concrete_compressive_strength_28d': '31 MPa', 'gwp_per_category_declared_unit': '327 kgCO2e', 'open_xpd_uuid': 'ec399r0r'}


In [ ]:
#filter down only the records that a value grater than 0 in the cementitious field
epd_records_filtered = [record for record in epd_records if 'cementitious' in record]

### Save to JSON

In [17]:
#Navigate up the directory and into the 01_raw_data folder to save the data
os.chdir(os.path.dirname(os.getcwd()))
os.chdir('01_raw_data')

with open('epddata_all.json', 'w') as f:
    json.dump(unique_records, f, indent=4)